In [1]:
%pip install aieng-agents

Note: you may need to restart the kernel to use updated packages.


In [2]:
from aieng.agents.tools import CodeInterpreter
from pathlib import Path

# --- CONFIGURATION ---
DB_PATH = "/data"
AGENT_LLM_NAMES = {
    "worker": "gemini-2.5-flash",
    "manager": "gemini-2.5-pro",
}

# --- SHARED TOOLS ---
local_db = Path(DB_PATH)
shared_code_interpreter = CodeInterpreter(
    template_name="lobsuu8phhb16r4r04yr", 
    # DEAD/REDUNDANT: old template kept for reference only
    # template_name="q1sg157kmhnqbfjth0ue", 
    local_files=[local_db] if local_db.exists() else []
)

In [3]:
from aieng.agents import setup_langfuse_tracer, set_up_logging
from dotenv import load_dotenv

load_dotenv()
set_up_logging()

# Setup tracing
tracer = setup_langfuse_tracer(service_name="my_agent_app")
# NOTE: tracer is initialized for observability but not used directly in later cells.

In [4]:
import json
from typing import Any

import sqlglot
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


E2B_SQLITE_PATH = "/data/data.db"
_ALLOWED_ROOTS = {"select", "union", "paren"}
_FORBIDDEN_NODES = {
    "create",
    "insert",
    "update",
    "delete",
    "drop",
    "alter",
    "truncate_table",
    "merge",
    "command",
    "pragma",
    "attach",
    "detach",
    "set",
}


def _normalize_code_interpreter_result(raw_result: Any) -> dict[str, Any]:
    if isinstance(raw_result, str):
        return json.loads(raw_result)
    if isinstance(raw_result, dict):
        return raw_result
    if hasattr(raw_result, "model_dump"):
        return raw_result.model_dump()
    raise TypeError(f"Unsupported E2B result type: {type(raw_result)!r}")


def _extract_stdout_text(raw_result: Any) -> str:
    payload = _normalize_code_interpreter_result(raw_result)
    error = payload.get("error")
    if error:
        raise RuntimeError(f"E2B execution failed: {error}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


def _validate_read_only_sql(query: str) -> str:
    cleaned_query = query.strip().rstrip(";")
    if not cleaned_query:
        raise ValueError("SQL query must not be empty.")

    expressions = sqlglot.parse(cleaned_query, read="sqlite")
    if len(expressions) != 1:
        raise ValueError("Only a single SQL statement is allowed.")

    expression = expressions[0]
    if expression.key.lower() not in _ALLOWED_ROOTS:
        raise ValueError("Only read-only SELECT queries are allowed.")

    for node in expression.walk():
        if node.key.lower() in _FORBIDDEN_NODES:
            raise ValueError(f"Forbidden SQL operation detected: {node.key}")

    return cleaned_query


async def get_database_schema() -> str:
    """Return tables and columns available in the E2B SQLite database."""
    code = f"""
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
rows = conn.execute(
    \"\"\"
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name
    \"\"\"
).fetchall()

sections = []
for name, relation_type in rows:
    table_info = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
    columns = ', '.join(f"{{column[1]}} {{column[2] or 'TEXT'}}" for column in table_info)
    sections.append(f"{{relation_type}}: {{name}}\\n  columns: {{columns}}")

print('\\n\\n'.join(sections) or 'No tables found.')
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


print(f"Core SQL helpers ready for {E2B_SQLITE_PATH}.")

Core SQL helpers ready for /data/data.db.


In [5]:
# CSV-scoped SQL helper for E2B: only rows whose transaction IDs are in the CSV are returned
import csv
import json
from pathlib import Path

import sqlglot
from sqlglot import expressions as exp


CSV_SCOPE_CANDIDATES = [
    Path("dataset_input_ids.csv"),
    Path("implementations/cibc-three/dataset_input_ids.csv"),
]


def _find_tx_csv_path() -> Path:
    for candidate in CSV_SCOPE_CANDIDATES:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find CSV file. Tried: {[str(p) for p in CSV_SCOPE_CANDIDATES]}"
    )


def _load_transaction_ids_from_csv(csv_path: Path) -> list[int]:
    with csv_path.open("r", newline="") as f:
        raw_lines = [line.strip() for line in f if line.strip()]

    if not raw_lines:
        raise ValueError(f"CSV file is empty: {csv_path}")

    first_token = raw_lines[0].split(",")[0].strip().lower()
    known_headers = {"transaction_id", "tx_id", "id"}

    if first_token in known_headers:
        with csv_path.open("r", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = [name.strip() for name in (reader.fieldnames or [])]
            id_column = "transaction_id" if "transaction_id" in fieldnames else fieldnames[0]
            source_ids = [row.get(id_column, "") for row in reader]
    else:
        source_ids = [line.split(",")[0].strip() for line in raw_lines]

    out = []
    for value in source_ids:
        text = str(value).strip()
        if not text:
            continue
        try:
            out.append(int(text))
        except ValueError:
            continue

    tx_ids = sorted(set(out))
    if not tx_ids:
        raise ValueError(f"No numeric transaction IDs found in {csv_path}")
    return tx_ids


async def _detect_transactions_key_column() -> str:
    code = f"""
import json
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
cols = [row[1] for row in conn.execute('PRAGMA table_info(transactions)').fetchall()]
conn.close()
print(json.dumps(cols))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    cols = json.loads(_extract_stdout_text(raw_result).strip())

    for candidate in ["transaction_id", "tx_id", "id", "transactionid"]:
        if candidate in cols:
            return candidate
    raise RuntimeError(f"Could not infer transactions key column. Found: {cols}")


def _scope_query_to_tx_ids(query: str, tx_ids: list[int], tx_col: str) -> str:
    parsed = sqlglot.parse_one(query, read="sqlite")

    aliases = []
    for table in parsed.find_all(exp.Table):
        if str(table.name).lower() == "transactions":
            aliases.append(table.alias_or_name or str(table.name))

    if not aliases:
        raise ValueError("Query must reference the transactions table so ID scoping can be applied.")

    in_values = ",".join(str(v) for v in tx_ids)
    alias_conditions = [
        sqlglot.parse_one(f"{alias}.{tx_col} IN ({in_values})", read="sqlite")
        for alias in aliases
    ]

    scope_condition = alias_conditions[0]
    for extra in alias_conditions[1:]:
        scope_condition = exp.or_(scope_condition, extra)

    if isinstance(parsed, exp.Select):
        where_clause = parsed.args.get("where")
        if where_clause is None:
            parsed.set("where", exp.Where(this=scope_condition))
        else:
            parsed.set("where", exp.Where(this=exp.and_(where_clause.this, scope_condition)))
        return parsed.sql(dialect="sqlite")

    return parsed.sql(dialect="sqlite")


async def run_csv_scoped_query(query: str, row_limit: int = 200) -> dict:
    """Run read-only SQL in E2B while forcing transaction_id scope to IDs from CSV."""
    safe_query = _validate_read_only_sql(query)

    csv_path = _find_tx_csv_path()
    tx_ids = _load_transaction_ids_from_csv(csv_path)
    tx_col = await _detect_transactions_key_column()
    scoped_query = _scope_query_to_tx_ids(safe_query, tx_ids, tx_col)

    code = f"""
import json
import sqlite3

query = {scoped_query!r}
conn = sqlite3.connect({E2B_SQLITE_PATH!r})
conn.row_factory = sqlite3.Row
cursor = conn.execute(query)
columns = [c[0] for c in (cursor.description or [])]
rows = [dict(r) for r in cursor.fetchmany({int(row_limit)})]
payload = {{
    'query': query,
    'columns': columns,
    'rows': rows,
    'row_limit': {int(row_limit)},
}}
print(json.dumps(payload, default=str))
conn.close()
"""

    raw_result = await shared_code_interpreter.run_code(code=code)
    return {
        "csv_path": str(csv_path),
        "scoped_transaction_ids": len(tx_ids),
        "transactions_key_column": tx_col,
        **json.loads(_extract_stdout_text(raw_result).strip()),
    }


In [6]:
# Natural-language agent that always executes SQL through CSV-scoped filtering

CSV_SCOPED_AGENT_INSTRUCTIONS = f"""
You are a SQLite analyst for an E2B database at {E2B_SQLITE_PATH}.

Mandatory workflow for each user question:
1. Call get_database_schema first.
2. Write a read-only SQL query based on the question.
3. Execute SQL ONLY via run_csv_scoped_query (never via run_sql_query).

Hard requirements:
- The result set must be restricted to transaction IDs from the CSV scope.
- Never invent tables or columns.
- Use only read-only SQL.

Response format:
ANSWER:
- Give the direct answer in plain English.

SQL:
```sql
<the exact SQL you executed>
```

RESULT:
- Summarize key rows/aggregates returned by the tool.
- Mention row-limit truncation when relevant.
""".strip()


csv_scoped_sql_agent = Agent(
    name="e2b_sql_agent_csv_scoped_nl",
    model=AGENT_LLM_NAMES["worker"],
    instruction=CSV_SCOPED_AGENT_INSTRUCTIONS,
    tools=[get_database_schema, run_csv_scoped_query],
)

csv_scoped_session_service = InMemorySessionService()
csv_scoped_runner = Runner(
    app_name=csv_scoped_sql_agent.name,
    agent=csv_scoped_sql_agent,
    session_service=csv_scoped_session_service,
)
csv_scoped_session = await csv_scoped_session_service.create_session(
    app_name=csv_scoped_sql_agent.name,
    user_id="notebook-user",
    state={},
)


async def reset_csv_scoped_agent_session() -> str:
    """Reset the CSV-scoped natural-language SQL agent session."""
    global csv_scoped_session
    csv_scoped_session = await csv_scoped_session_service.create_session(
        app_name=csv_scoped_sql_agent.name,
        user_id="notebook-user",
        state={},
    )
    return csv_scoped_session.id


async def ask_csv_scoped_agent(question: str) -> str:
    """Ask a natural-language question; SQL execution is always CSV transaction-ID scoped."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in csv_scoped_runner.run_async(
        user_id="notebook-user",
        session_id=csv_scoped_session.id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


print(f"CSV-scoped NL SQL agent ready. Session: {csv_scoped_session.id}")
print("Use: await ask_csv_scoped_agent('your question in natural language')")

CSV-scoped NL SQL agent ready. Session: e533d007-66ec-40de-8ff5-b60dc12969e5
Use: await ask_csv_scoped_agent('your question in natural language')


In [7]:
response = await ask_csv_scoped_agent(
    "Show top 20 highest amount transactions with client_id, transaction_id, and merchant_state."
)
print(response)

/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/runners.py:1481: DeprecationWarning: deprecated
  save_input_blobs_as_artifacts=run_config.save_input_blobs_as_artifacts,


17:12:17.562 invocation
17:12:17.565   invoke_agent e2b_sql_agent_csv_scoped_nl
17:12:17.568     call_llm


/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/genai/_api_client.py:755: DeprecationWarning: Inheritance class AiohttpClientSession from ClientSession is discouraged
  class AiohttpClientSession(aiohttp.ClientSession):  # type: ignore[misc]


17:12:18.713       execute_tool get_database_schema


17:12:19.711     call_llm
17:12:21.126       execute_tool run_csv_scoped_query


17:12:25.389     call_llm
ANSWER:
The top 20 transactions with the highest amounts within the provided CSV scope all have an amount of 0.

SQL:

```sql
SELECT
  client_id,
  transaction_id,
  merchant_state,
  CAST(amount AS REAL) AS amount
FROM transactions
ORDER BY
  amount DESC
LIMIT 20;
```


RESULT:
The query returned 20 rows, each showing a `client_id`, `transaction_id`, `merchant_state`, and an `amount`. All 20 transactions displayed an `amount` of 0. For example, one of the transactions is with `client_id`: 909, `transaction_id`: 7476248, `merchant_state`: None, and `amount`: 0.


In [8]:
# DEAD CELL: intentionally left as a spacer during iterative notebook development.

In [9]:
# ========== FRAUD DETECTION ENGINE (SCHEMA-CORRECTED) ==========
import numpy as np
from datetime import datetime, timedelta
from collections import Counter
from typing import Optional


class FraudDetectionEngine:
    """
    Multi-factor fraud detection system using statistical analysis,
    velocity checks, merchant patterns, and anomaly scoring.
    """

    def __init__(self, z_score_threshold: float = 2.5):
        """
        Args:
            z_score_threshold: Number of std devs from mean to flag as anomaly (default: 2.5)
        """
        self.z_score_threshold = z_score_threshold

    async def analyze_transaction_anomalies(self, transactions: list[dict]) -> dict[str, any]:
        """
        Detect statistical anomalies in transaction amounts.
        Returns transactions flagged by amount deviation.
        """
        amounts = [float(t.get('amount', 0)) for t in transactions if t.get('amount')]
        if len(amounts) < 3:
            return {'flagged': [], 'stats': {}}

        mean = np.mean(amounts)
        std = np.std(amounts)
        
        anomalies = []
        for t in transactions:
            amt = float(t.get('amount', 0))
            if std > 0:
                z_score = abs((amt - mean) / std)
                if z_score > self.z_score_threshold:
                    anomalies.append({
                        'transaction_id': t.get('transaction_id'),
                        'amount': amt,
                        'z_score': z_score,
                        'deviation_pct': ((amt - mean) / mean * 100) if mean > 0 else 0,
                        'fraud_risk': 'HIGH' if z_score > 3.5 else 'MEDIUM'
                    })

        return {
            'flagged': sorted(anomalies, key=lambda x: x['z_score'], reverse=True),
            'stats': {
                'mean_amount': round(mean, 2),
                'std_deviation': round(std, 2),
                'min_amount': min(amounts),
                'max_amount': max(amounts),
                'anomaly_count': len(anomalies),
                'anomaly_percentage': round(len(anomalies) / len(amounts) * 100, 2)
            }
        }

    async def analyze_velocity(self, transactions: list[dict], time_window_minutes: int = 60) -> dict:
        """
        Detect high-frequency transaction velocity (multiple txns in short timeframe).
        Flags potential card-not-present fraud or rapid-fire attacks.
        """
        if not transactions or len(transactions) < 2:
            return {'flagged': [], 'stats': {}}

        # Parse timestamps and group by client
        by_client = {}
        for t in transactions:
            client_id = t.get('client_id')
            if not client_id:
                continue
            
            # Extract transaction date
            tx_date = t.get('date') or t.get('transaction_date')
            
            if client_id not in by_client:
                by_client[client_id] = []
            by_client[client_id].append({
                'transaction_id': t.get('transaction_id'),
                'date': tx_date,
                'amount': float(t.get('amount', 0)),
                'merchant_id': t.get('merchant_id', 'Unknown')
            })

        # Detect velocity patterns
        velocity_flags = []
        for client_id, txns in by_client.items():
            if len(txns) >= 3:
                # Simple velocity: 3+ transactions in time window
                velocity_flags.append({
                    'client_id': client_id,
                    'transaction_count': len(txns),
                    'transactions': [t['transaction_id'] for t in txns],
                    'fraud_risk': 'HIGH' if len(txns) >= 5 else 'MEDIUM',
                    'total_amount': sum(t['amount'] for t in txns)
                })

        return {
            'flagged': velocity_flags,
            'stats': {
                'clients_analyzed': len(by_client),
                'high_velocity_clients': len(velocity_flags),
                'avg_txn_per_client': round(len(transactions) / len(by_client)) if by_client else 0
            }
        }

    async def analyze_merchant_patterns(self, transactions: list[dict]) -> dict:
        """
        Detect suspicious merchant patterns using merchant_id:
        - Repeat merchant IDs (card reuse at same location)
        - Geographic clustering (merchant_id in multiple states)
        """
        merchant_counter = Counter()
        state_counter = Counter()
        merchant_states = {}  # merchant_id -> set of states
        merchant_txns = {}    # merchant_id -> list of transaction_ids

        for t in transactions:
            merchant_id = t.get('merchant_id')
            state = t.get('merchant_state', 'Unknown')
            amount = float(t.get('amount', 0))

            if merchant_id is None:
                continue

            merchant_counter[merchant_id] += 1
            state_counter[state] += 1

            if merchant_id not in merchant_states:
                merchant_states[merchant_id] = set()
                merchant_txns[merchant_id] = []
            
            merchant_states[merchant_id].add(state)
            merchant_txns[merchant_id].append(t.get('transaction_id'))

        # Flag merchant IDs used multiple times (possible card reuse)
        suspicious_merchants = [
            {
                'merchant_id': m,
                'frequency': count,
                'states': list(merchant_states[m]),
                'transactions': merchant_txns[m][:10],  # First 10 txns
                'fraud_risk': 'HIGH' if count >= 3 else 'MEDIUM'
            }
            for m, count in merchant_counter.most_common(10) if count >= 2
        ]

        # Flag merchant IDs with multi-state presence (geographic red flag)
        geographic_anomalies = [
            {
                'merchant_id': m,
                'states': len(states),
                'state_list': list(states),
                'transactions': merchant_txns[m][:10],
                'fraud_risk': 'HIGH'
            }
            for m, states in merchant_states.items() if len(states) >= 2
        ]

        return {
            'repeat_merchants': suspicious_merchants,
            'geographic_anomalies': geographic_anomalies,
            'stats': {
                'unique_merchants': len(merchant_counter),
                'unique_states': len(state_counter),
                'repeat_merchant_count': len(suspicious_merchants),
                'geographic_anomaly_count': len(geographic_anomalies),
                'most_common_merchant': merchant_counter.most_common(1)[0] if merchant_counter else ('N/A', 0)
            }
        }

    async def calculate_fraud_score(self, 
                                    transactions: list[dict],
                                    anomaly_results: dict,
                                    velocity_results: dict,
                                    merchant_results: dict) -> list[dict]:
        """
        Combine multiple fraud signals into a composite fraud risk score (0-100).
        Higher scores = higher fraud likelihood.
        """
        tx_by_id = {t.get('transaction_id'): t for t in transactions}
        
        fraud_scores = {}

        # Initialize all transactions with base score
        for tx_id in tx_by_id.keys():
            fraud_scores[tx_id] = {'score': 20, 'flags': [], 'details': {}}

        # Add points for amount anomalies
        for anomaly in anomaly_results.get('flagged', []):
            tx_id = anomaly['transaction_id']
            if tx_id in fraud_scores:
                points = 35 if anomaly['fraud_risk'] == 'HIGH' else 20
                fraud_scores[tx_id]['score'] += points
                fraud_scores[tx_id]['flags'].append(f"Amount Anomaly (z={anomaly['z_score']:.2f})")
                fraud_scores[tx_id]['details']['amount_anomaly'] = anomaly

        # Add points for velocity flags
        for velocity in velocity_results.get('flagged', []):
            client_id = velocity['client_id']
            points = 25 if velocity['fraud_risk'] == 'HIGH' else 15
            for tx_id in velocity['transactions']:
                if tx_id in fraud_scores:
                    fraud_scores[tx_id]['score'] += points
                    fraud_scores[tx_id]['flags'].append(f"High Velocity ({velocity['transaction_count']} txns)")
                    fraud_scores[tx_id]['details']['velocity'] = velocity

        # Add points for merchant anomalies
        for tx in transactions:
            tx_id = tx.get('transaction_id')
            merchant_id = tx.get('merchant_id')
            
            # Check if merchant_id is in repeat list
            for repeat_m in merchant_results.get('repeat_merchants', []):
                if repeat_m['merchant_id'] == merchant_id and tx_id in fraud_scores:
                    fraud_scores[tx_id]['score'] += 15
                    fraud_scores[tx_id]['flags'].append(f"Repeat Merchant ({repeat_m['frequency']} uses)")

            # Check if merchant_id has geographic red flags
            for geo_a in merchant_results.get('geographic_anomalies', []):
                if geo_a['merchant_id'] == merchant_id and tx_id in fraud_scores:
                    fraud_scores[tx_id]['score'] += 20
                    fraud_scores[tx_id]['flags'].append(f"Multi-State Merchant ({len(geo_a['states'])} states)")

        # Convert to list and categorize
        result = []
        for tx_id, data in fraud_scores.items():
            score = min(100, data['score'])  # Cap at 100
            
            if score >= 70:
                risk_level = 'CRITICAL'
            elif score >= 50:
                risk_level = 'HIGH'
            elif score >= 30:
                risk_level = 'MEDIUM'
            else:
                risk_level = 'LOW'

            result.append({
                'transaction_id': tx_id,
                'fraud_score': score,
                'risk_level': risk_level,
                'flags': data['flags'],
                'transaction': tx_by_id[tx_id]
            })

        return sorted(result, key=lambda x: x['fraud_score'], reverse=True)


# Initialize the fraud detection engine
fraud_engine = FraudDetectionEngine(z_score_threshold=2.5)
print("✓ Fraud Detection Engine initialized and ready (schema corrected)")


async def run_fraud_analysis(question: str) -> str:
    """
    Execute fraud detection analysis with comprehensive reporting.
    """
    # Step 1: Get transaction data using the NL agent
    print(f"\n📊 Analyzing: {question}")
    print("-" * 60)
    
    # Step 2: Run fraud detection
    # Get base transactions data
    result = await run_csv_scoped_query(
        "SELECT * FROM transactions LIMIT 500",
        row_limit=500
    )
    
    transactions = result.get('rows', [])
    print(f"Loaded {len(transactions)} transactions for analysis")
    
    # Run all fraud detection analyses in parallel
    anomalies = await fraud_engine.analyze_transaction_anomalies(transactions)
    velocity = await fraud_engine.analyze_velocity(transactions)
    merchants = await fraud_engine.analyze_merchant_patterns(transactions)
    fraud_scores = await fraud_engine.calculate_fraud_score(transactions, anomalies, velocity, merchants)
    
    # Step 3: Format and return comprehensive report
    critical_fraud = [tx for tx in fraud_scores if tx['risk_level'] == 'CRITICAL']
    high_fraud = [tx for tx in fraud_scores if tx['risk_level'] == 'HIGH']
    
    report = f"""
FRAUD DETECTION ANALYSIS REPORT
{'='*60}

📌 CRITICAL FINDINGS
  • Critical Risk Transactions: {len(critical_fraud)}
  • High Risk Transactions: {len(high_fraud)}
  • Total Transactions Analyzed: {len(transactions)}

💰 AMOUNT ANOMALIES
  • Flagged: {anomalies['stats']['anomaly_count']}
  • Mean Amount: ${anomalies['stats']['mean_amount']}
  • Std Dev: ${anomalies['stats']['std_deviation']}
  • Anomaly Rate: {anomalies['stats']['anomaly_percentage']}%

⚡ VELOCITY ANALYSIS
  • Clients Analyzed: {velocity['stats']['clients_analyzed']}
  • High-Velocity Clients: {velocity['stats']['high_velocity_clients']}
  • Avg Txn per Client: {velocity['stats']['avg_txn_per_client']}

🏪 MERCHANT PATTERNS
  • Unique Merchants: {merchants['stats']['unique_merchants']}
  • Unique States: {merchants['stats']['unique_states']}
  • Repeat Merchants Flagged: {merchants['stats']['repeat_merchant_count']}
  • Geographic Anomalies: {merchants['stats']['geographic_anomaly_count']}

🚨 TOP 10 HIGHEST RISK TRANSACTIONS
"""
    
    for i, tx in enumerate(fraud_scores[:10], 1):
        report += f"\n  {i}. TX_ID: {tx['transaction_id']} | Score: {tx['fraud_score']}/100 | Risk: {tx['risk_level']}"
        for flag in tx['flags'][:2]:
            report += f"\n     • {flag}"
    
    return report


print("✓ Fraud analysis function ready")
print("   Usage: await run_fraud_analysis('your question about fraud')")


✓ Fraud Detection Engine initialized and ready (schema corrected)
✓ Fraud analysis function ready
   Usage: await run_fraud_analysis('your question about fraud')


In [10]:
# ========== FRAUD DETECTION TOOLS FOR AGENT (FIXED FOR ACTUAL DB SCHEMA) ==========


def _parse_amount(amount_str: str) -> float:
    """Parse amount string that may contain currency formatting ($99.92)."""
    if not amount_str:
        return 0.0
    # Strip currency symbols and whitespace, then convert
    cleaned = str(amount_str).replace('$', '').replace(',', '').strip()
    try:
        return float(cleaned)
    except (ValueError, TypeError):
        return 0.0


async def detect_suspicious_transactions_sql(max_z_score: float = 2.5) -> dict:
    """Tool: Get statistical outliers by transaction amount."""
    query = """
    SELECT 
        transaction_id,
        amount,
        client_id,
        merchant_id,
        merchant_state,
        date,
        mcc
    FROM transactions
    ORDER BY amount DESC
    LIMIT 100
    """
    result = await run_csv_scoped_query(query, row_limit=100)
    rows = result.get('rows', [])

    # query_runner = run_full_db_query if scope_mode == "full_db" else run_csv_scoped_query
    # result = await query_runner(query, row_limit=100)
    
    # Calculate statistics - with currency format handling
    amounts = [_parse_amount(r['amount']) for r in rows if r.get('amount')]
    if len(amounts) < 3:
        return {'error': 'Insufficient data for analysis'}
    
    mean = np.mean(amounts)
    std = np.std(amounts)
    
    outliers = []
    for r in rows:
        if std > 0:
            amount_val = _parse_amount(r['amount'])
            z_score = abs((amount_val - mean) / std)
            if z_score > max_z_score:
                outliers.append({
                    'transaction_id': r['transaction_id'],
                    'amount': amount_val,
                    'client_id': r['client_id'],
                    'merchant_id': r.get('merchant_id'),
                    'merchant_state': r.get('merchant_state'),
                    'z_score': round(z_score, 2),
                    'deviation_pct': round((amount_val - mean) / mean * 100, 2),
                    'mcc': r.get('mcc')
                })
    
    return {
        'outliers': sorted(outliers, key=lambda x: x['amount'], reverse=True),
        'statistics': {
            'mean_amount': round(mean, 2),
            'std_deviation': round(std, 2),
            'outlier_count': len(outliers)
        }
    }


async def detect_high_velocity_clients_sql() -> dict:
    """Tool: Identify clients with high transaction frequency."""
    query = """
    SELECT 
        client_id,
        COUNT(*) as transaction_count,
        SUM(CAST(amount AS REAL)) as total_amount,
        AVG(CAST(amount AS REAL)) as avg_amount,
        COUNT(DISTINCT merchant_id) as unique_merchants,
        COUNT(DISTINCT merchant_state) as unique_states
    FROM transactions
    GROUP BY client_id
    HAVING transaction_count >= 3
    ORDER BY transaction_count DESC
    LIMIT 50
    """
    result = await run_csv_scoped_query(query, row_limit=50)
    
    velocity_flags = []
    for row in result.get('rows', []):
        risk = 'CRITICAL' if row.get('transaction_count', 0) >= 7 else \
               'HIGH' if row.get('transaction_count', 0) >= 5 else 'MEDIUM'
        
        velocity_flags.append({
            'client_id': row['client_id'],
            'transaction_count': row['transaction_count'],
            'total_amount': _parse_amount(row['total_amount']) if row.get('total_amount') else 0,
            'avg_amount': round(_parse_amount(row['avg_amount']), 2) if row.get('avg_amount') else 0,
            'unique_merchants': row['unique_merchants'],
            'unique_states': row['unique_states'],
            'risk_level': risk
        })
    
    return {
        'high_velocity_clients': velocity_flags,
        'total_flagged': len(velocity_flags)
    }


async def detect_geographic_anomalies_sql() -> dict:
    """Tool: Find merchant_ids appearing in multiple states (geographic red flag)."""
    query = """
    SELECT 
        merchant_id,
        COUNT(DISTINCT merchant_state) as state_count,
        GROUP_CONCAT(DISTINCT merchant_state) as states,
        COUNT(*) as frequency
    FROM transactions
    GROUP BY merchant_id
    HAVING state_count > 1
    ORDER BY frequency DESC
    LIMIT 50
    """
    result = await run_csv_scoped_query(query, row_limit=50)
    
    anomalies = []
    for row in result.get('rows', []):
        states = str(row.get('states', '')).split(',') if row.get('states') else []
        anomalies.append({
            'merchant_id': row['merchant_id'],
            'state_count': row['state_count'],
            'states': states,
            'frequency': row['frequency'],
            'risk_level': 'HIGH' if row['state_count'] >= 3 else 'MEDIUM'
        })
    
    return {
        'geographic_anomalies': anomalies,
        'total_anomalies': len(anomalies)
    }


async def detect_repeat_merchants_sql() -> dict:
    """Tool: Identify merchant_ids with multiple transactions (possible card reuse)."""
    query = """
    SELECT 
        merchant_id,
        COUNT(*) as frequency,
        COUNT(DISTINCT client_id) as unique_clients,
        SUM(CAST(amount AS REAL)) as total_amount,
        AVG(CAST(amount AS REAL)) as avg_amount
    FROM transactions
    GROUP BY merchant_id
    HAVING frequency >= 2
    ORDER BY frequency DESC
    LIMIT 50
    """
    result = await run_csv_scoped_query(query, row_limit=50)
    
    repeats = []
    for row in result.get('rows', []):
        risk = 'CRITICAL' if row['frequency'] >= 8 else \
               'HIGH' if row['frequency'] >= 5 else 'MEDIUM'
        
        repeats.append({
            'merchant_id': row['merchant_id'],
            'frequency': row['frequency'],
            'unique_clients': row['unique_clients'],
            'total_amount': _parse_amount(row['total_amount']) if row.get('total_amount') else 0,
            'avg_amount': round(_parse_amount(row['avg_amount']), 2) if row.get('avg_amount') else 0,
            'risk_level': risk
        })
    
    return {
        'repeat_merchants': repeats,
        'total_repeat_merchants': len(repeats)
    }


# Create Fraud-Specific Agent
FRAUD_AGENT_INSTRUCTIONS = """You are a fraud detection expert analyzing transaction data.

Your role:
1. Investigate suspicious transaction patterns
2. Identify fraud indicators: amount anomalies, velocity, geographic inconsistencies
3. Provide actionable fraud risk assessments
4. Use multiple detection tools for comprehensive analysis

Available tools:
- detect_suspicious_transactions_sql: Statistical outliers by amount (z-score analysis)
- detect_high_velocity_clients_sql: Clients with rapid transaction frequency
- detect_geographic_anomalies_sql: Merchant IDs appearing in multiple states
- detect_repeat_merchants_sql: Frequently-used merchant IDs
- run_csv_scoped_query: Direct SQL for custom queries

Data schema:
- transactions table: transaction_id, date, client_id, card_id, amount, use_chip, merchant_id, merchant_city, merchant_state, zip, mcc, errors

Fraud risk levels:
- CRITICAL: >70 risk score or extreme anomalies
- HIGH: 50-70 risk score or multiple fraud indicators
- MEDIUM: 30-50 risk score or single significant indicator
- LOW: <30 risk score or normal transactions

Response format:
ANALYSIS:
<detailed findings>

FRAUD INDICATORS:
- [key findings about fraud patterns detected]

RISK ASSESSMENT:
- [summary of risk levels and affected transactions]

RECOMMENDATIONS:
- [actionable recommendations for fraud prevention]
"""

fraud_detection_agent = Agent(
    name="fraud_detection_expert",
    model=AGENT_LLM_NAMES["manager"],
    instruction=FRAUD_AGENT_INSTRUCTIONS,
    tools=[
        detect_suspicious_transactions_sql,
        detect_high_velocity_clients_sql,
        detect_geographic_anomalies_sql,
        detect_repeat_merchants_sql,
        run_csv_scoped_query,
    ],
)

fraud_session_service = InMemorySessionService()
fraud_runner = Runner(
    app_name=fraud_detection_agent.name,
    agent=fraud_detection_agent,
    session_service=fraud_session_service,
)

fraud_session = await fraud_session_service.create_session(
    app_name=fraud_detection_agent.name,
    user_id="notebook-user",
    state={},
)


# async def ask_fraud_agent(question: str) -> str:
#     """Ask the fraud detection agent to investigate suspicious patterns."""
#     content = types.Content(role="user", parts=[types.Part(text=question)])
#     final_chunks: list[str] = []

#     async for event in fraud_runner.run_async(
#         user_id="notebook-user",
#         session_id=fraud_session.id,
#         new_message=content,
#     ):
#         if not event.is_final_response() or not event.content:
#             continue
#         for part in event.content.parts:
#             if part.text:
#                 final_chunks.append(part.text)

#     return "\n".join(chunk for chunk in final_chunks if chunk).strip()

async def ask_fraud_agent(question: str, fresh_session: bool = True) -> str:
    session_id = fraud_session.id
    if fresh_session:
        new_session = await fraud_session_service.create_session(
            app_name=fraud_detection_agent.name,
            user_id="notebook-user",
            state={},
        )
        session_id = new_session.id

    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in fraud_runner.run_async(
        user_id="notebook-user",
        session_id=session_id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


print("✓ Fraud Detection Agent ready (schema corrected, currency format handling added)")
print("   Usage: await ask_fraud_agent('Identify the most suspicious transactions')")


✓ Fraud Detection Agent ready (schema corrected, currency format handling added)
   Usage: await ask_fraud_agent('Identify the most suspicious transactions')


In [11]:
# ========== FRAUD DETECTION EXAMPLES ==========

# Example 1: Get the actual fraud transactions using the fraud agent
print("🔍 Running comprehensive fraud detection analysis...")
print("=" * 70)

fraud_analysis = await ask_fraud_agent(
    """Identify the top fraud transactions in our dataset. 
    Focus on:
    1. Transactions with unusual amounts (statistical outliers)
    2. Clients making rapid-fire purchases (velocity fraud)
    3. Merchant IDs appearing across multiple states (geographic anomalies)
    4. Repeat merchant IDs that might indicate testing or card reuse
    
    Provide specific transaction IDs, risk scores, and actionable insights."""
)
print("FRAUD DETECTION RESULTS")
print("=" * 70)
print(fraud_analysis)


🔍 Running comprehensive fraud detection analysis...
17:12:29.381 invocation
17:12:29.382   invoke_agent fraud_detection_expert
17:12:29.384     call_llm
17:12:33.528       execute_tool detect_suspicious_transactions_sql
17:12:33.569       execute_tool detect_high_velocity_clients_sql
17:12:33.611       execute_tool detect_geographic_anomalies_sql
17:12:33.654       execute_tool detect_repeat_merchants_sql


17:12:41.948       execute_tool (merged)
17:12:41.952     call_llm
FRAUD DETECTION RESULTS
ANALYSIS:
A multi-faceted fraud detection analysis was conducted, focusing on amount anomalies, transaction velocity, geographic inconsistencies, and merchant reuse. The investigation utilized four distinct detection tools to identify a range of suspicious activities.

1.  **Amount Anomaly Detection:**
    A single transaction was identified as a significant statistical outlier. Transaction `10366997` for `$973.40` is an extreme anomaly, with a z-score of `9.95` and a deviation of over `800%` from the mean transaction amount.

2.  **High-Velocity Client Detection:**
    A total of 50 clients were flagged for high-velocity transactions, all at a 'CRITICAL' risk level. The most active client, `ID 909`, conducted 247 transactions. These clients exhibit patterns of rapid purchasing across numerous merchants and states, a common indicator of automated card testing or enumeration attacks.

3.  **Geogra

In [12]:
# REDUNDANT TEST CALL: superseded by later full_db + direct tool examples.
# q_alt = "List only the top 5 suspicious transaction IDs and one-line reason for each."
# r3 = await ask_fraud_agent(q_alt, fresh_session=True)
# print(r3[:700])

In [ ]:
# LEGACY/DUPLICATE BLOCK (kept commented for reference):
# This cell redefines several functions and re-creates the fraud agent with dual-scope support.
# It duplicates behavior defined earlier and can cause confusion by shadowing prior definitions.
# Keep disabled unless you explicitly want the dual-scope variant.
#
# Prompt redundancy note: this cell also re-declares FRAUD_AGENT_INSTRUCTIONS
# (the same variable name used earlier), which overwrites the first prompt in runtime.
#
# Full-DB + CSV-scoped fraud tools and agent (paste into one new code cell)
# import json
# import numpy as np
# import json
# from typing import Any
#
# import sqlglot
# from google.adk.agents import Agent
# from google.adk.runners import Runner
# from google.adk.sessions import InMemorySessionService
# from google.genai import types
#
#
# async def run_full_db_query(query: str, row_limit: int = 200) -> dict:
#     """Run read-only SQL in E2B against the full DB (no CSV transaction ID filter)."""
#     safe_query = _validate_read_only_sql(query)
#
#     code = f"""
# import json
# import sqlite3
#
# query = {safe_query!r}
# conn = sqlite3.connect({E2B_SQLITE_PATH!r})
# conn.row_factory = sqlite3.Row
# cursor = conn.execute(query)
# columns = [c[0] for c in (cursor.description or [])]
# rows = [dict(r) for r in cursor.fetchmany({int(row_limit)})]
# payload = {{
#     'query': query,
#     'columns': columns,
#     'rows': rows,
#     'row_limit': {int(row_limit)},
# }}
# print(json.dumps(payload, default=str))
# conn.close()
# """
#     raw_result = await shared_code_interpreter.run_code(code=code)
#     return {
#         "scope": "full_db",
#         **json.loads(_extract_stdout_text(raw_result).strip()),
#     }
#
#
# def _parse_amount(amount_str: str) -> float:
#     if not amount_str:
#         return 0.0
#     cleaned = str(amount_str).replace("$", "").replace(",", "").strip()
#     try:
#         return float(cleaned)
#     except (ValueError, TypeError):
#         return 0.0
#
#
# def _query_runner(scope_mode: str):
#     return run_full_db_query if scope_mode == "full_db" else run_csv_scoped_query
#
#
# async def detect_suspicious_transactions_sql(
#     max_z_score: float = 2.5,
#     scope_mode: str = "csv",
# ) -> dict:
#     query = """
#     SELECT
#         transaction_id,
#         amount,
#         client_id,
#         merchant_id,
#         merchant_state,
#         date,
#         mcc
#     FROM transactions
#     ORDER BY CAST(amount AS REAL) DESC
#     LIMIT 100
#     """
#     result = await _query_runner(scope_mode)(query, row_limit=100)
#     rows = result.get("rows", [])
#
#     amounts = [_parse_amount(r.get("amount")) for r in rows if r.get("amount") is not None]
#     if len(amounts) < 3:
#         return {"error": "Insufficient data for analysis", "scope_mode": scope_mode}
#
#     mean = np.mean(amounts)
#     std = np.std(amounts)
#
#     outliers = []
#     for r in rows:
#         if std > 0:
#             amount_val = _parse_amount(r.get("amount"))
#             z_score = abs((amount_val - mean) / std)
#             if z_score > max_z_score:
#                 outliers.append(
#                     {
#                         "transaction_id": r.get("transaction_id"),
#                         "amount": amount_val,
#                         "client_id": r.get("client_id"),
#                         "merchant_id": r.get("merchant_id"),
#                         "merchant_state": r.get("merchant_state"),
#                         "z_score": round(z_score, 2),
#                         "deviation_pct": round((amount_val - mean) / mean * 100, 2) if mean else 0,
#                         "mcc": r.get("mcc"),
#                     }
#                 )
#
#     return {
#         "scope_mode": scope_mode,
#         "scope": result.get("scope", "csv"),
#         "outliers": sorted(outliers, key=lambda x: x["amount"], reverse=True),
#         "statistics": {
#             "mean_amount": round(mean, 2),
#             "std_deviation": round(std, 2),
#             "outlier_count": len(outliers),
#         },
#     }
#
#
# async def detect_high_velocity_clients_sql(scope_mode: str = "csv") -> dict:
#     query = """
#     SELECT
#         client_id,
#         COUNT(*) as transaction_count,
#         SUM(CAST(amount AS REAL)) as total_amount,
#         AVG(CAST(amount AS REAL)) as avg_amount,
#         COUNT(DISTINCT merchant_id) as unique_merchants,
#         COUNT(DISTINCT merchant_state) as unique_states
#     FROM transactions
#     GROUP BY client_id
#     HAVING transaction_count >= 3
#     ORDER BY transaction_count DESC
#     LIMIT 50
#     """
#     result = await _query_runner(scope_mode)(query, row_limit=50)
#
#     velocity_flags = []
#     for row in result.get("rows", []):
#         tx_count = row.get("transaction_count", 0)
#         risk = "CRITICAL" if tx_count >= 7 else "HIGH" if tx_count >= 5 else "MEDIUM"
#         velocity_flags.append(
#             {
#                 "client_id": row.get("client_id"),
#                 "transaction_count": tx_count,
#                 "total_amount": _parse_amount(row.get("total_amount")) if row.get("total_amount") else 0,
#                 "avg_amount": round(_parse_amount(row.get("avg_amount")), 2) if row.get("avg_amount") else 0,
#                 "unique_merchants": row.get("unique_merchants"),
#                 "unique_states": row.get("unique_states"),
#                 "risk_level": risk,
#             }
#         )
#
#     return {
#         "scope_mode": scope_mode,
#         "scope": result.get("scope", "csv"),
#         "high_velocity_clients": velocity_flags,
#         "total_flagged": len(velocity_flags),
#     }
#
#
# async def detect_geographic_anomalies_sql(scope_mode: str = "csv") -> dict:
#     query = """
#     SELECT
#         merchant_id,
#         COUNT(DISTINCT merchant_state) as state_count,
#         GROUP_CONCAT(DISTINCT merchant_state) as states,
#         COUNT(*) as frequency
#     FROM transactions
#     GROUP BY merchant_id
#     HAVING state_count > 1
#     ORDER BY frequency DESC
#     LIMIT 50
#     """
#     result = await _query_runner(scope_mode)(query, row_limit=50)
#
#     anomalies = []
#     for row in result.get("rows", []):
#         states = str(row.get("states", "")).split(",") if row.get("states") else []
#         anomalies.append(
#             {
#                 "merchant_id": row.get("merchant_id"),
#                 "state_count": row.get("state_count"),
#                 "states": states,
#                 "frequency": row.get("frequency"),
#                 "risk_level": "HIGH" if (row.get("state_count") or 0) >= 3 else "MEDIUM",
#             }
#         )
#
#     return {
#         "scope_mode": scope_mode,
#         "scope": result.get("scope", "csv"),
#         "geographic_anomalies": anomalies,
#         "total_anomalies": len(anomalies),
#     }
#
#
# async def detect_repeat_merchants_sql(scope_mode: str = "csv") -> dict:
#     query = """
#     SELECT
#         merchant_id,
#         COUNT(*) as frequency,
#         COUNT(DISTINCT client_id) as unique_clients,
#         SUM(CAST(amount AS REAL)) as total_amount,
#         AVG(CAST(amount AS REAL)) as avg_amount
#     FROM transactions
#     GROUP BY merchant_id
#     HAVING frequency >= 2
#     ORDER BY frequency DESC
#     LIMIT 50
#     """
#     result = await _query_runner(scope_mode)(query, row_limit=50)
#
#     repeats = []
#     for row in result.get("rows", []):
#         freq = row.get("frequency", 0)
#         risk = "CRITICAL" if freq >= 8 else "HIGH" if freq >= 5 else "MEDIUM"
#         repeats.append(
#             {
#                 "merchant_id": row.get("merchant_id"),
#                 "frequency": freq,
#                 "unique_clients": row.get("unique_clients"),
#                 "total_amount": _parse_amount(row.get("total_amount")) if row.get("total_amount") else 0,
#                 "avg_amount": round(_parse_amount(row.get("avg_amount")), 2) if row.get("avg_amount") else 0,
#                 "risk_level": risk,
#             }
#         )
#
#     return {
#         "scope_mode": scope_mode,
#         "scope": result.get("scope", "csv"),
#         "repeat_merchants": repeats,
#         "total_repeat_merchants": len(repeats),
#     }
#
#
# FRAUD_AGENT_INSTRUCTIONS = """You are a fraud detection expert analyzing transaction data.
#
# Scope handling:
# - Default to scope_mode='csv' (selected transaction IDs only).
# - If the user asks for full database analysis, call tools with scope_mode='full_db'.
# - Explicitly mention which scope you used.
# """
#
# fraud_detection_agent = Agent(
#     name="fraud_detection_expert",
#     model=AGENT_LLM_NAMES["manager"],
#     instruction=FRAUD_AGENT_INSTRUCTIONS,
#     tools=[
#         detect_suspicious_transactions_sql,
#         detect_high_velocity_clients_sql,
#         detect_geographic_anomalies_sql,
#         detect_repeat_merchants_sql,
#         run_csv_scoped_query,
#         run_full_db_query,
#     ],
# )
#
# fraud_session_service = InMemorySessionService()
# fraud_runner = Runner(
#     app_name=fraud_detection_agent.name,
#     agent=fraud_detection_agent,
#     session_service=fraud_session_service,
# )
#
# fraud_session = await fraud_session_service.create_session(
#     app_name=fraud_detection_agent.name,
#     user_id="notebook-user",
#     state={},
# )
#
#
# async def ask_fraud_agent(question: str, fresh_session: bool = True) -> str:
#     session_id = fraud_session.id
#     if fresh_session:
#         new_session = await fraud_session_service.create_session(
#             app_name=fraud_detection_agent.name,
#             user_id="notebook-user",
#             state={},
#         )
#         session_id = new_session.id
#
#     content = types.Content(role="user", parts=[types.Part(text=question)])
#     final_chunks: list[str] = []
#
#     async for event in fraud_runner.run_async(
#         user_id="notebook-user",
#         session_id=session_id,
#         new_message=content,
#     ):
#         if not event.is_final_response() or not event.content:
#             continue
#         for part in event.content.parts:
#             if part.text:
#                 final_chunks.append(part.text)
#
#     return "\n".join(chunk for chunk in final_chunks if chunk).strip()
#
#
# print("Fraud agent ready with dual scope support.")
# print("Use scope wording like: 'analyze full database' or 'analyze selected CSV transactions'.")

In [14]:
# REDUNDANT EXECUTION EXAMPLE: depends on legacy duplicate dual-scope block above.
# await ask_fraud_agent("Use full_db scope and return only 5 transaction IDs.", fresh_session=True)

In [15]:
# REDUNDANT EXECUTION EXAMPLE: direct full_db tool calls rely on the legacy duplicate block.
# s = await detect_suspicious_transactions_sql(scope_mode="full_db")
# v = await detect_high_velocity_clients_sql(scope_mode="full_db")
# g = await detect_geographic_anomalies_sql(scope_mode="full_db")
# r = await detect_repeat_merchants_sql(scope_mode="full_db")
#
# print("Outlier txns:", len(s.get("outliers", [])))
# print("High velocity clients:", len(v.get("high_velocity_clients", [])))
# print("Geographic anomalies:", len(g.get("geographic_anomalies", [])))
# print("Repeat merchants:", len(r.get("repeat_merchants", [])))
# print("Top 5 outlier transaction IDs:", [x["transaction_id"] for x in s.get("outliers", [])[:5]])